# Cloud LoRA train - MLX-LM -> HF PEFT port (Qwen2.5, free Colab T4)

Faithful port of our local MLX-LM LoRA run (`configs/lora-real.yaml`) onto the CUDA/HuggingFace
path. Same LoRA recipe; only the hardware (M1 -> T4) and library (MLX-LM -> transformers+PEFT)
change - so the resulting adapter is a like-for-like before/after against the MLX adapter on the
same frozen holdout (evaluated locally back on the Mac).

**Note on stack:** we use the plain `transformers.Trainer` + `peft` directly rather than
`trl.SFTTrainer`. trl's SFTTrainer is a thin wrapper over exactly this Trainer + a completion-only
collator; doing it explicitly is more transparent for learning and dodges trl's version churn.
Still pure HF PEFT, still not unsloth.

**Faithful-port mapping (verified against mlx_lm source, not assumed):**

| MLX (`lora-real.yaml`) | here | why |
|---|---|---|
| `scale: 20.0` | `lora_alpha=160` | MLX scale is a DIRECT multiplier; PEFT uses `alpha/r`, so alpha = scale*r = 160 |
| all 7 linears/block | `target_modules=[q,k,v,o,gate,up,down]_proj` | MLX default LoRA-ifies every linear, not just q/v |
| `num_layers: 8` | top-8 decoder blocks (auto) | MLX uses `model.layers[-8:]` |
| `rank 8`, `dropout 0` | `r=8`, `lora_dropout=0.0` | 1:1 |
| 4-bit base | bnb nf4 QLoRA | quant SCHEME differs (affine vs NormalFloat-4) - a port caveat |
| `iters 100`, `save_every 50` | `max_steps=100`, `save_steps=50` | keep BOTH iter-50 + iter-100 (step f compared both) |

**Run order:** GPU check -> install -> config -> load 4-bit base -> upload data -> tokenize
(completion-only) -> 2-step SMOKE -> 100-step REAL -> download adapters.

**Data discipline:** only `train.jsonl` + `valid.jsonl` go up (CC0 Civil Comments, which originates
from Google anyway). `holdout.jsonl` + `test.jsonl` stay on the Mac, evaluated locally only.

## 1. GPU check + install

In [1]:
!nvidia-smi -L || echo "NO GPU - set Runtime > Change runtime type > T4 GPU, then Run all again"
!pip -q install -U "transformers>=4.45,<4.48" "peft>=0.13,<0.15" "bitsandbytes>=0.44" "accelerate>=1.0" "datasets>=3.0"

GPU 0: Tesla T4 (UUID: GPU-9818c0fb-20a7-3b84-cd89-17c9402cf929)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 90.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.8/374.8 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 99.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6

## 2. Config - THE ONE-LINE SWITCH for experiment #2 lives here

In [2]:
# === EXPERIMENT SWITCH ===============================================
# 1.5B faithful port first. For experiment #2 ("does capacity beat the
# DATA ceiling?") change ONLY this line to "Qwen/Qwen2.5-7B-Instruct"
# and Run all again - the layer range auto-adjusts from the model config.
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
# ====================================================================

# Faithful-port LoRA recipe (matches configs/lora-real.yaml + mlx_lm defaults):
LORA_R        = 8
LORA_ALPHA    = 160     # MLX scale(20) is a DIRECT multiplier => alpha = scale * r = 160
LORA_DROPOUT  = 0.0
N_TOP_LAYERS  = 8       # MLX num_layers: 8  (LoRA on the top-8 decoder blocks)
TARGET_MODULES = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]
MAX_SEQ_LEN   = 512
MAX_STEPS     = 100
SAVE_STEPS    = 50      # -> checkpoints at 50 and 100
LEARNING_RATE = 1e-4
SEED          = 0

## 3. Load the 4-bit base (QLoRA) + tokenizer

In [3]:
import torch, gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",            # NormalFloat-4 (MLX used affine-4bit: a port caveat)
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16, # T4 is Turing -> float16, NOT bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def build_model():
    """Fresh 4-bit base + LoRA adapters. Called separately for smoke and real
    so the real run never starts from the smoke's 2 steps."""
    m = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, quantization_config=bnb, device_map="auto", torch_dtype=torch.float16,
    )
    m = prepare_model_for_kbit_training(m, use_gradient_checkpointing=True)
    n = m.config.num_hidden_layers
    layer_range = list(range(n - N_TOP_LAYERS, n))   # top-8, auto for 1.5B(28) or 7B(28)
    lora = LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        target_modules=TARGET_MODULES, layers_to_transform=layer_range,
        layers_pattern="layers", bias="none", task_type="CAUSAL_LM",
    )
    m = get_peft_model(m, lora)
    m.print_trainable_parameters()
    print("LoRA on layers", layer_range, "of", n)
    return m

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## 4. Upload the two frozen-split files

In [4]:
from google.colab import files
print("Pick data/real/prepared/train.jsonl AND valid.jsonl from your Mac:")
files.upload()   # writes them into the working directory

import json
def load_jsonl(p):
    with open(p) as f:
        return [json.loads(l) for l in f if l.strip()]
train_rows = load_jsonl("train.jsonl")
valid_rows = load_jsonl("valid.jsonl")
print(len(train_rows), "train rows,", len(valid_rows), "valid rows")

Pick data/real/prepared/train.jsonl AND valid.jsonl from your Mac:


Saving train.jsonl to train.jsonl
Saving valid.jsonl to valid.jsonl
112 train rows, 24 valid rows


## 5. Tokenize with completion-only masking

Loss only on the assistant's answer token(s) - the system+user+header are masked with `-100`.
This matches how MLX-LM trains chat data (don't train the model to reproduce the prompt).

In [5]:
from datasets import Dataset

def tokenize(ex):
    msgs = ex["messages"]
    full   = tokenizer.apply_chat_template(msgs,       tokenize=False, add_generation_prompt=False)
    prompt = tokenizer.apply_chat_template(msgs[:-1],  tokenize=False, add_generation_prompt=True)
    full_ids   = tokenizer(full,   truncation=True, max_length=MAX_SEQ_LEN, add_special_tokens=False)["input_ids"]
    prompt_ids = tokenizer(prompt, truncation=True, max_length=MAX_SEQ_LEN, add_special_tokens=False)["input_ids"]
    labels = list(full_ids)
    for i in range(min(len(prompt_ids), len(labels))):
        labels[i] = -100   # completion-only: no loss before the answer
    return {"input_ids": full_ids, "attention_mask": [1]*len(full_ids), "labels": labels}

train_ds = Dataset.from_list(train_rows).map(tokenize, remove_columns=["messages"])
valid_ds = Dataset.from_list(valid_rows).map(tokenize, remove_columns=["messages"])
print("row0 unmasked answer tokens:", sum(1 for x in train_ds[0]["labels"] if x != -100),
      "(should be small - just the label word + end token)")

Map:   0%|          | 0/112 [00:00<?, ? examples/s]

Map:   0%|          | 0/24 [00:00<?, ? examples/s]

row0 unmasked answer tokens: 3 (should be small - just the label word + end token)


## 6. Trainer factory (shared by smoke + real)

In [6]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

collator = DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100)

def make_trainer(model, max_steps, out_dir, do_eval):
    args = TrainingArguments(
        output_dir=out_dir,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        max_steps=max_steps,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="constant",     # MLX used a constant lr
        warmup_steps=0,
        logging_steps=10,
        save_strategy="steps", save_steps=SAVE_STEPS,
        eval_strategy=("steps" if do_eval else "no"), eval_steps=SAVE_STEPS,
        fp16=True,                        # T4
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        optim="adamw_torch",
        seed=SEED, report_to="none", save_total_limit=None,
    )
    return Trainer(model=model, args=args, train_dataset=train_ds,
                   eval_dataset=(valid_ds if do_eval else None), data_collator=collator)

## 7. SMOKE - 2 steps to prove the pipeline parses (cheapest-first)

Fresh model, 2 steps, discarded. If this errors we fix it here, not 100 steps in.

In [7]:
smoke = build_model()
make_trainer(smoke, max_steps=2, out_dir="/tmp/smoke", do_eval=False).train()
print("\nSMOKE OK - pipeline runs, loss is finite. Discarding smoke model.")
del smoke; gc.collect(); torch.cuda.empty_cache()

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 2,637,824 || all params: 1,546,352,128 || trainable%: 0.1706
LoRA on layers [20, 21, 22, 23, 24, 25, 26, 27] of 28


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Step,Training Loss



SMOKE OK - pipeline runs, loss is finite. Discarding smoke model.


## 8. REAL - 100-step faithful-port train (saves iter-50 + iter-100)

In [8]:
model = build_model()
trainer = make_trainer(model, max_steps=MAX_STEPS, out_dir="out", do_eval=True)
trainer.train()
print("\nREAL done. Adapters saved under out/checkpoint-50 and out/checkpoint-100.")

trainable params: 2,637,824 || all params: 1,546,352,128 || trainable%: 0.1706
LoRA on layers [20, 21, 22, 23, 24, 25, 26, 27] of 28


Step,Training Loss,Validation Loss
50,0.664900,0.189902
100,1.579600,0.719226



REAL done. Adapters saved under out/checkpoint-50 and out/checkpoint-100.


## 9. Package both adapters + provenance, download

In [9]:
import os, shutil, json, datetime

os.makedirs("adapter-peft", exist_ok=True)
for ckpt in (SAVE_STEPS, MAX_STEPS):
    dst = f"adapter-peft/adapter-iter{ckpt}"
    os.makedirs(dst, exist_ok=True)
    for fn in ("adapter_config.json", "adapter_model.safetensors"):
        shutil.copy(f"out/checkpoint-{ckpt}/{fn}", f"{dst}/{fn}")

meta = {
    "base_model": BASE_MODEL, "lora_r": LORA_R, "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT, "target_modules": TARGET_MODULES,
    "n_top_layers": N_TOP_LAYERS, "max_steps": MAX_STEPS, "save_steps": SAVE_STEPS,
    "learning_rate": LEARNING_RATE, "seed": SEED,
    "quant": "bnb-nf4-double", "compute_dtype": "float16",
    "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
}
with open("adapter-peft/adapter-meta.json", "w") as f:
    json.dump(meta, f, indent=2)
print(json.dumps(meta, indent=2))

shutil.make_archive("adapter-peft", "zip", "adapter-peft")
from google.colab import files
files.download("adapter-peft.zip")   # hand this back to Claude in the repo working dir

{
  "base_model": "Qwen/Qwen2.5-1.5B-Instruct",
  "lora_r": 8,
  "lora_alpha": 160,
  "lora_dropout": 0.0,
  "target_modules": [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj"
  ],
  "n_top_layers": 8,
  "max_steps": 100,
  "save_steps": 50,
  "learning_rate": 0.0001,
  "seed": 0,
  "quant": "bnb-nf4-double",
  "compute_dtype": "float16",
  "created_utc": "2026-09-04T09:26:30.835666+00:00"
}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>